<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Background Subtraction</b></h1>
</div>

## Context

Neurointerventional fluoroscopy combines a largely static anatomical background with sparse moving guidewires and microcatheters whose visibility can be limited by low contrast, acquisition noise, residual anatomy, and field-of-view boundaries.

The dataset contains ten fluoroscopic frames with paired guidewire and microcatheter annotations. The project treats foreground extraction as an experimental model-selection problem: the original first-frame background pipeline is established as a baseline, then background modeling, radiometric normalization, spatial filtering, spectral filtering, morphology, segmentation, and field-of-view restriction are evaluated through controlled ablations.

## Problem Statement

Develop and validate a background-subtraction pipeline for automatic enhancement and binary localization of moving guidewires and microcatheters in the supplied fluoroscopic sequence.

For every evaluated frame, the retained pipeline must produce a binary mask using the laboratory convention:

- `0` = moving tool;
- `1` = background.

The study must:

- reproduce and quantify the original first-frame-reference baseline;
- evaluate a temporal-median background model;
- stabilize radiometric normalization across the sequence;
- optimize spatial Gaussian and spectral high-pass filtering;
- evaluate morphological refinement and segmentation-threshold sensitivity;
- compare deterministic thresholding against a two-component EM/GMM alternative;
- constrain processing to a valid field of view without removing annotated tool pixels;
- assemble one explicit retained processing configuration;
- report per-frame and sequence-level SAD, MSE, and PSNR;
- generate representative masks, overlays, parameter-sensitivity plots, and metric trajectories;
- validate numerical consistency, annotation coverage, mask semantics, and output completeness.

Non-rigid registration, optical flow, learned segmentation, adaptive online background models, and clinical deployment are outside scope.

## Inputs and Fixed Parameters

| Item | Fixed value / convention |
| --- | --- |
| Frame identifiers | 201, 211, ..., 291 |
| Number of frames | 10 |
| Fluoroscopy image | `data/catheter/frame_<id>.png` |
| Guidewire annotation | `data/catheter/<id>_GuideWire.tiff` |
| Microcatheter annotation | `data/catheter/<id>_MicroCath.tiff` |
| Ground truth | union of guidewire + microcatheter annotations |
| Mask convention | `0 = tool`, `1 = background` |
| Baseline spatial Gaussian | $\sigma=1.0$ |
| Baseline pre-subtraction saturation | percentiles 0–90 |
| Baseline post-dilation saturation | percentiles 10–100 |
| Baseline dilation | disk radius 2 |
| Baseline threshold | $T=0.1$ |
| Baseline opening | disk radius 2 |
| Spatial sigma candidates | 0.5, 0.75, 1.0, 1.25, 1.5, 2.0 |
| Spectral HPF cutoffs | 5, 7.5, 10, 12.5, 20, 40, 80 |
| Dilation radii | 0, 1, 2, 3, 4 |
| Opening radii | 0, 1, 2, 3, 4 |
| Coarse threshold candidates | 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.20, 0.25 |
| Fine threshold step | 0.005 within ±0.03 of best coarse threshold |
| GMM components | 2 |
| GMM fit sample | at most 50,000 pixels |
| GMM random state | 0 |
| FOV threshold candidates | 120, 125, 130, 135, 140, 150, 160, 180, 200 |
| Primary metrics | SAD, MSE, PSNR |
| Model-selection criterion | minimum mean MSE, with guardrails |
| Required outputs | 17 diagnostic PNG figures |

Only one processing decision may change during each ablation stage. A change is retained only when its sequence-level evidence supports it and all validity guardrails remain satisfied.

## 1. Validate Data and Ground-Truth Paths

For every frame identifier $t\in\{201,211,\ldots,291\}$:

1. verify the presence of the fluoroscopy frame;
2. verify both guidewire and microcatheter annotation files;
3. load all three arrays;
4. reduce annotation images to one channel when required;
5. verify identical spatial dimensions.

Construct the combined ground-truth mask using the supplied annotation convention and preserve the laboratory definition

$$
0=\text{tool},
\qquad
1=\text{background}.
$$

**Required evidence**

- complete 10-frame / 20-annotation inventory;
- common image dimensions;
- one representative image + guidewire + microcatheter + union visualization saved as `01_representative_data.png`.

Abort on any missing or shape-inconsistent file.

## 2. Reproduce the Original Baseline Pipeline

Reproduce the original laboratory pipeline without modification.

For each frame:

1. normalize grayscale image values to $[0,1]$;
2. apply Gaussian smoothing with $\sigma=1.0$;
3. apply per-frame percentile saturation 0–90;
4. use processed frame 201 as the background reference;
5. for frames 211–291, compute the signed residual

$$
R_t=-(I_t-B_{201});
$$

6. dilate with disk radius 2;
7. apply percentile saturation 10–100;
8. threshold at $T=0.1$;
9. apply opening with disk radius 2;
10. invert to the required `0=tool, 1=background` convention;
11. compute SAD, MSE, and PSNR against ground truth.

Ground truth is thickened using the original disk-radius-2 operation exactly as in the baseline.

Report per-frame and mean±standard-deviation metrics for the nine evaluable frames.

Save representative baseline results and preserve them as the reference for all later ablations.

## 3. Replace the First-Frame Background with a Temporal Median

Change only the background model.

Keep fixed:

- Gaussian $\sigma=1.0$;
- per-frame 0–90 saturation;
- signed subtraction direction;
- dilation radius 2;
- post-dilation 10–100 saturation;
- threshold 0.1;
- opening radius 2.

Replace

$$
B=I_{201}
$$

with the pixelwise temporal median

$$
B_{med}(x,y)
=
\operatorname{median}_t I_t(x,y).
$$

Evaluate:

1. the same nine frames 211–291 for fair comparison with the baseline;
2. frame 201 separately, since the temporal model now permits its evaluation.

Save `02_background_models.png`.

**Decision rule:** retain the temporal median only if sequence-level metrics improve or remain competitive while enabling a physically more representative background model.

## 4. Stabilize the Pre-Subtraction Histogram Transformation

The per-frame 0–90 percentile mapping can introduce artificial temporal variation. Replace it with one fixed mapping estimated from the temporal-median background.

Procedure:

1. Gaussian-filter every frame using the currently retained $\sigma$;
2. compute the temporal median in that Gaussian-smoothed domain;
3. estimate fixed lower/upper levels from background percentiles 0 and 90;
4. apply the same mapping to every frame:

$$
I'=
\operatorname{clip}
\left(
\frac{I-L}{H-L},
0,1
\right).
$$

All downstream operations remain unchanged.

Save `03_histogram_transformation.png`.

Compare fixed mapping vs per-frame mapping on all 10 frames using mean SAD/MSE/PSNR.

Retain the fixed mapping only if the evidence supports improved temporal consistency.

## 5. Optimize Spatial Gaussian Filtering

Evaluate

$$
\sigma\in
\{0.5,0.75,1.0,1.25,1.5,2.0\}.
$$

For each candidate:

1. Gaussian-filter every frame;
2. rebuild the temporal-median background in the same filtered domain;
3. recompute the fixed background-derived intensity mapping;
4. hold all later stages constant;
5. evaluate the full sequence.

Save:

- `05_spatial_filtering.png`;
- `06_spatial_sigma_sensitivity.png`.

Report mean SAD, MSE, and PSNR for every $\sigma$.

**Selection rule:** retain the $\sigma$ with minimum mean MSE, provided visual inspection confirms that thin tool responses are not excessively suppressed.

## 6. Add and Tune Spectral-Domain High-Pass Filtering

After temporal subtraction, introduce a Gaussian high-pass filter

$$
H(u,v)
=
1-
\exp
\left(
-\frac{D(u,v)^2}{2D_0^2}
\right).
$$

Evaluate

$$
D_0\in
\{5,7.5,10,12.5,20,40,80\}.
$$

For each cutoff:

1. compute 2-D FFT of the retained residual;
2. center the spectrum;
3. multiply by $H(u,v)$;
4. inverse shift and inverse FFT;
5. retain the positive residual response;
6. apply the unchanged downstream morphology and segmentation;
7. compute sequence-level metrics.

Save `07_spectral_filtering.png`.

**Selection rule:** choose minimum mean MSE and retain spectral filtering only if it improves on the no-spectral-filter configuration.

## 7. Optimize Morphological Refinement

Evaluate morphology while all retained upstream stages remain fixed.

Test

$$
r_d\in\{0,1,2,3,4\},
\qquad
r_o\in\{0,1,2,3,4\},
$$

where radius 0 means the corresponding operation is skipped.

Procedure:

1. sweep dilation radius while opening remains fixed at the baseline value;
2. fix the best dilation radius and sweep opening radius;
3. evaluate the complete $5\times5$ pair grid as a consistency check.

Report mean SAD/MSE/PSNR for every tested pair.

Save `10_morphological_refinement.png`.

**Selection rule:** minimum mean MSE, subject to preserving the sparse guidewire/microcatheter structures without unnecessary expansion.

## 8. Optimize the Segmentation Threshold

Re-optimize the deterministic threshold after all retained preprocessing changes.

### Coarse search

Evaluate

$$
T\in
\{0.02,0.04,0.06,0.08,0.10,0.12,0.14,0.16,0.20,0.25\}.
$$

Select the coarse candidate with minimum mean MSE.

### Fine search

Evaluate thresholds from

$$
T_{coarse}^{*}-0.03
$$

to

$$
T_{coarse}^{*}+0.03
$$

using step 0.005, clipped to valid positive thresholds.

For every threshold, report mean SAD/MSE/PSNR.

Save:

- `08_segmentation.png`;
- `09_threshold_sensitivity.png`.

The validated retained value must be stated explicitly. In the current experiment the optimum is expected near the original 0.1 value and is selected only from the measured sweep.

## 9. Compare Threshold Segmentation with EM/GMM

Replace only the final deterministic decision rule with a two-component Gaussian Mixture Model.

Use:

- $K=2$ Gaussian components;
- full covariance;
- deterministic random state 0;
- at most 50,000 sampled pixels for fitting;
- prediction over every pixel after fitting.

Model:

$$
p(x)
=
\sum_{k=1}^{2}
\pi_k
\mathcal N(x\mid\mu_k,\Sigma_k).
$$

Interpret the higher-mean component as tool-like response.

Compare threshold and GMM on identical pre-segmentation feature images using:

- per-frame SAD/MSE/PSNR;
- sequence means;
- representative masks.

The more complex GMM is retained only if it outperforms the deterministic threshold under the same evidence criteria.

## 10. Restrict Processing to a Valid Field-of-View Mask

Construct a fixed field-of-view mask from the raw temporal-median background.

Evaluate threshold candidates

$$
T_{FOV}\in
\{120,125,130,135,140,150,160,180,200\}.
$$

For each candidate, compute:

- FOV area fraction;
- minimum ground-truth coverage across the sequence;
- mean SAD/MSE/PSNR after applying the FOV constraint.

Ground-truth coverage is

$$
C_t=
\frac{|Q\cap G_t|}
{|G_t|}.
$$

A candidate is **safe** only if

$$
\min_t C_t\ge0.999999.
$$

Among safe candidates, select the one with minimum mean MSE.

This guardrail prevents artificial metric improvement by masking annotated tool pixels.

## 11. Assemble the Final Retained Pipeline

State the final configuration as one explicit ordered pipeline containing:

1. retained background model;
2. fixed intensity mapping;
3. retained Gaussian $\sigma$;
4. signed temporal residual;
5. retained Gaussian HPF cutoff;
6. retained dilation/opening radii;
7. retained deterministic threshold or GMM decision;
8. retained FOV threshold;
9. final mask convention.

Report every retained numerical parameter in one table.

Also list the rejected alternatives and the evidence-based reason for rejection.

Save representative intermediate stages including `04_background_residual.png`.

## 12. Compute Sequence-Level Quantitative Evaluation

For every frame, compare final mask $M_t$ with ground truth $G_t$ using

$$
\mathrm{SAD}_t
=
\frac1N
\sum_i
|G_{t,i}-M_{t,i}|,
$$

$$
\mathrm{MSE}_t
=
\frac1N
\sum_i
(G_{t,i}-M_{t,i})^2,
$$

$$
\mathrm{PSNR}_t
=
20\log_{10}
\left(
\frac{L}{\sqrt{\mathrm{MSE}_t}}
\right).
$$

Report per-frame values plus mean and standard deviation.

Generate and save:

- `11_mask_vs_ground_truth.png`;
- `12_guidance_overlay.png`;
- `13_validation_overlay.png`;
- `14_sad_vs_time.png`;
- `15_mse_vs_time.png`;
- `16_psnr_vs_time.png`;
- `17_overlap_vs_time.png`.

Use the overlays to identify false detections and missed tool regions that aggregate metrics may hide.

## 13. Run Numerical and Output-file Validation Checks

Verify all of the following:

1. exactly 10 frame/annotation triplets are available;
2. all frames and masks have identical dimensions;
3. all final predictions are binary and use `0=tool, 1=background`;
4. SAD/MSE/PSNR arrays contain 10 finite values;
5. retained $\sigma>0$;
6. retained spectral cutoff $D_0>0$;
7. retained threshold satisfies $0<T<1$;
8. morphology radii belong to the tested candidate sets;
9. retained FOV satisfies minimum ground-truth coverage $\ge0.999999$;
10. GMM comparison uses deterministic settings;
11. all 17 required diagnostic figures exist;
12. the reported final configuration matches the parameters actually used for evaluation.

Any failed condition invalidates the final result until corrected.

## Completion Criterion

The project is complete when the original baseline has been reproduced, each subsequent modification has been evaluated as a controlled ablation, retained and rejected decisions are supported by sequence-level evidence, the final threshold/FOV branch satisfies annotation-coverage guardrails, all ten frames have complete metrics, and the 17 required diagnostic outputs are reproducible from the stated final configuration.